# Proyecto .- Micro-Pipeline UCC en R
## Micro-pipeline de análisis exploratorio de datos (EDA) en R base
- Nombre : David León Salas
- Numero de control : 202680124
- Correo : 202680124@ucc.mx
- Fecha: 16 de octubre de 2026

In [33]:
# Datos proporcionados (muestra anonimizada de satisfacción)
calificaciones <- c(4, 5, 3, 5, 4, 2, 1, 5, 4, 5, 3, 4, 4, 5, 1, 2, 5, 5)
comentarios <- c(
  "Muy buen curso, volvería a tomarlo",
  "Excelente contenido y profesor",
  "Regular, esperaba más ejercicios",
  "Muy completo, realmente excelente",
  "Aprendí bastante, buen curso",
  "Demasiado básico para mi nivel",
  "Muy malo, no lo recomiendo",
  "Excelente, aprendí cosas nuevas",
  "Muy útil para mi trabajo",
  "Genial enfoque, excelente!",
  "Regular, temas poco profundizados",
  "Curso bueno, aunque mejorable",
  "Muy buen curso, buenos ejemplos",
  "Excelente de verdad, recomendable",
  "Malo, esperaba otra cosa",
  "Podría mejorar bastante",
  "Excelente, superó mis expectativas",
  "Excelente, volvería a tomarlo"
)

cat("Tipo de calificaciones:", class(calificaciones), "\n")
cat("Tipo de comentarios:", class(comentarios), "\n")
cat("Cantidad de calificaciones:", length(calificaciones), "\n")
cat("Cantidad de comentarios:", length(comentarios), "\n")

Tipo de calificaciones: numeric 
Tipo de comentarios: character 
Cantidad de calificaciones: 18 
Cantidad de comentarios: 18 


# 1. Funciones auxiliares

Todas las funciones son **puras**: reciben sus datos como argumentos, no modifican variables globales y siempre devuelven el mismo resultado para la misma entrada.

### 1.1. Funciones estadísticas

In [34]:
validar_datos <- function(datos) {
  if (length(datos) == 0) {
    stop("El vector no puede estar vacío")
  }
}

media <- function(datos) {
  validar_datos(datos)
  total <- sum(datos)
  cantidad <- length(datos)
  return(total / cantidad)
}

mediana <- function(datos) {
  validar_datos(datos)
  ordenados <- sort(datos)
  cantidad <- length(ordenados)
  centro <- cantidad %/% 2

  if (cantidad %% 2 == 0) {
    primero <- ordenados[centro]
    segundo <- ordenados[centro + 1]
    return((primero + segundo) / 2)
  } else {
    return(ordenados[centro + 1])
  }
}

moda <- function(datos) {
  validar_datos(datos)
  conteos <- table(datos)
  mayor_frecuencia <- max(conteos)
  modas <- names(conteos)[conteos == mayor_frecuencia]
  return(as.numeric(modas))
}

rango <- function(datos) {
  validar_datos(datos)
  mayor <- max(datos)
  menor <- min(datos)
  return(mayor - menor)
}

varianza_poblacional <- function(datos) {
  promedio <- media(datos)
  diferencias <- datos - promedio      # operación vectorizada
  suma_cuadrados <- sum(diferencias^2)
  return(suma_cuadrados / length(datos))
}

frecuencias <- function(datos) {
  absoluta <- table(datos)
  relativa <- prop.table(absoluta) * 100
  data.frame(
    calificacion = as.numeric(names(absoluta)),
    frecuencia = as.integer(absoluta),
    porcentaje = round(as.numeric(relativa), 2),
    frecuencia_acumulada = cumsum(as.integer(absoluta)),
    porcentaje_acumulado = round(cumsum(as.numeric(relativa)), 2)
  )
}

### 1.2. Funciones de procesamiento de texto

In [35]:
limpiar_texto <- function(texto) {
  texto <- tolower(texto)
  texto <- chartr("áéíóúü", "aeiouu", texto)   # elimina acentos
  texto <- gsub("[[:punct:]¡¿]", " ", texto)   # elimina signos de puntuación
  texto <- gsub("\\s+", " ", texto)            # colapsa espacios repetidos
  return(trimws(texto))
}

tokenizar <- function(textos) {
  return(strsplit(textos, " "))
}

top_palabras <- function(tokens, k = 5) {
  conteo <- table(tokens)
  tabla <- data.frame(
    palabra = names(conteo),
    frecuencia = as.integer(conteo)
  )
  # orden descendente por frecuencia y alfabético en caso de empate
  tabla <- tabla[order(-tabla$frecuencia, tabla$palabra), ]
  rownames(tabla) <- NULL
  return(head(tabla, k))
}

contar_palabras <- function(tokens, vocabulario) {
  return(sum(tokens %in% vocabulario))
}

porcentaje_palabras <- function(tokens, vocabulario) {
  if (length(tokens) == 0) {
    return(0)
  }
  return(contar_palabras(tokens, vocabulario) / length(tokens) * 100)
}

### 1.3. Función de clasificación

Reglas definidas:
1. Si el comentario contiene una negación explícita de recomendación (`no lo recomiendo` / `no recomiendo`) → **Negativo**.
2. Si tiene palabras positivas **y no** tiene negativas → **Positivo**.
3. Si tiene palabras negativas **y no** tiene positivas → **Negativo**.
4. En cualquier otro caso (mezcla de ambas o ninguna) → **Neutro**.

In [36]:
clasificar_comentario <- function(comentario, positivas, negativas) {
  limpio <- limpiar_texto(comentario)
  tokens <- tokenizar(limpio)[[1]]

  tiene_positivas <- contar_palabras(tokens, positivas) > 0
  tiene_negativas <- contar_palabras(tokens, negativas) > 0
  frase <- paste0(" ", limpio, " ")
  negacion <- grepl(" no lo recomiendo ", frase, fixed = TRUE) ||
              grepl(" no recomiendo ", frase, fixed = TRUE)

  if (negacion) {
    return("Negativo")
  } else if (tiene_positivas && !tiene_negativas) {
    return("Positivo")
  } else if (tiene_negativas && !tiene_positivas) {
    return("Negativo")
  } else {
    return("Neutro")
  }
}

cat("Funciones auxiliares cargadas correctamente.\n")

Funciones auxiliares cargadas correctamente.


# 2. Métricas estadísticas

### 2.1. Métricas centrales y de dispersión

In [37]:
cat(sprintf("Media: %.4f\n", media(calificaciones)))
cat("Calificaciones ordenadas:", sort(calificaciones), "\n")
cat("Mediana:", mediana(calificaciones), "\n")
cat("Moda:", moda(calificaciones), "\n")
cat("Rango:", rango(calificaciones), "\n")
cat(sprintf("Varianza poblacional: %.4f\n", varianza_poblacional(calificaciones)))

Media: 3.7222
Calificaciones ordenadas: 1 1 2 2 3 3 4 4 4 4 4 5 5 5 5 5 5 5 
Mediana: 4 
Moda: 5 
Rango: 4 
Varianza poblacional: 1.8673


### 2.2. Validación contra funciones de R base

In [38]:
n <- length(calificaciones)

resumen <- data.frame(
  metrica = c("Media", "Mediana", "Moda", "Rango", "Varianza poblacional"),
  funcion_propia = c(
    media(calificaciones),
    mediana(calificaciones),
    moda(calificaciones)[1],
    rango(calificaciones),
    varianza_poblacional(calificaciones)
  ),
  r_base = c(
    mean(calificaciones),
    median(calificaciones),
    as.numeric(names(which.max(table(calificaciones)))),
    diff(range(calificaciones)),
    var(calificaciones) * (n - 1) / n   # var() es muestral; se ajusta a poblacional
  )
)
resumen$coincide <- abs(resumen$funcion_propia - resumen$r_base) < 1e-9

print(resumen, digits = 5)

               metrica funcion_propia r_base coincide
1                Media         3.7222 3.7222     TRUE
2              Mediana         4.0000 4.0000     TRUE
3                 Moda         5.0000 5.0000     TRUE
4                Rango         4.0000 4.0000     TRUE
5 Varianza poblacional         1.8673 1.8673     TRUE


### 2.3. Distribución de frecuencias de calificaciones

In [39]:
print(table(calificaciones))
cat("\n")

distribucion <- frecuencias(calificaciones)
print(distribucion, row.names = FALSE)
cat("\n")

for (i in seq_len(nrow(distribucion))) {
  cat(sprintf("Calificación %d: %d estudiantes (%.2f%%)\n",
              distribucion$calificacion[i],
              distribucion$frecuencia[i],
              distribucion$porcentaje[i]))
}

calificaciones
1 2 3 4 5 
2 2 2 5 7 

 calificacion frecuencia porcentaje frecuencia_acumulada porcentaje_acumulado
            1          2      11.11                    2                11.11
            2          2      11.11                    4                22.22
            3          2      11.11                    6                33.33
            4          5      27.78                   11                61.11
            5          7      38.89                   18               100.00

Calificación 1: 2 estudiantes (11.11%)
Calificación 2: 2 estudiantes (11.11%)
Calificación 3: 2 estudiantes (11.11%)
Calificación 4: 5 estudiantes (27.78%)
Calificación 5: 7 estudiantes (38.89%)


# 3. Procesamiento de texto

### 3.1. Conversión a minúsculas y eliminación de acentos y signos de puntuación

In [40]:
# limpiar_texto es vectorizada: procesa todo el vector de comentarios a la vez
comentarios_limpios <- limpiar_texto(comentarios)

cat("Original:", comentarios[1], "\n")
cat("Limpio:  ", comentarios_limpios[1], "\n\n")
print(comentarios_limpios)

Original: Muy buen curso, volvería a tomarlo 
Limpio:   muy buen curso volveria a tomarlo 

 [1] "muy buen curso volveria a tomarlo" "excelente contenido y profesor"   
 [3] "regular esperaba mas ejercicios"   "muy completo realmente excelente" 
 [5] "aprendi bastante buen curso"       "demasiado basico para mi nivel"   
 [7] "muy malo no lo recomiendo"         "excelente aprendi cosas nuevas"   
 [9] "muy util para mi trabajo"          "genial enfoque excelente"         
[11] "regular temas poco profundizados"  "curso bueno aunque mejorable"     
[13] "muy buen curso buenos ejemplos"    "excelente de verdad recomendable" 
[15] "malo esperaba otra cosa"           "podria mejorar bastante"          
[17] "excelente supero mis expectativas" "excelente volveria a tomarlo"     


### 3.2. Tokenización y conjunto de palabras únicas

In [41]:
tokens_por_comentario <- tokenizar(comentarios_limpios)
palabras <- unlist(tokens_por_comentario)
palabras_unicas <- sort(unique(palabras))   # equivalente a un set

cat("Palabras del primer comentario:", tokens_por_comentario[[1]], "\n")
cat("Total de palabras:", length(palabras), "\n")
cat("Cantidad de palabras distintas:", length(palabras_unicas), "\n")
cat("Palabras únicas:\n")
print(palabras_unicas)

Palabras del primer comentario: muy buen curso volveria a tomarlo 
Total de palabras: 76 
Cantidad de palabras distintas: 51 
Palabras únicas:
 [1] "a"             "aprendi"       "aunque"        "basico"       
 [5] "bastante"      "buen"          "bueno"         "buenos"       
 [9] "completo"      "contenido"     "cosa"          "cosas"        
[13] "curso"         "de"            "demasiado"     "ejemplos"     
[17] "ejercicios"    "enfoque"       "esperaba"      "excelente"    
[21] "expectativas"  "genial"        "lo"            "malo"         
[25] "mas"           "mejorable"     "mejorar"       "mi"           
[29] "mis"           "muy"           "nivel"         "no"           
[33] "nuevas"        "otra"          "para"          "poco"         
[37] "podria"        "profesor"      "profundizados" "realmente"    
[41] "recomendable"  "recomiendo"    "regular"       "supero"       
[45] "temas"         "tomarlo"       "trabajo"       "util"         
[49] "verdad"        "volveri

### 3.3. Top 5 de palabras más frecuentes

In [42]:
top_5 <- top_palabras(palabras, 5)
print(top_5)

    palabra frecuencia
1 excelente          7
2       muy          5
3     curso          4
4      buen          3
5         a          2


### 3.4. Vocabularios y porcentaje de palabras positivas y negativas

In [43]:
positivas <- c(
  "buen", "bueno", "buenos", "excelente", "completo", "aprendi",
  "util", "genial", "recomendable", "recomiendo", "supero"
)
negativas <- c("malo", "basico", "mejorable", "mejorar")

cantidad_positivas <- contar_palabras(palabras, positivas)
cantidad_negativas <- contar_palabras(palabras, negativas)

cat("Apariciones positivas:", cantidad_positivas, "\n")
cat("Apariciones negativas:", cantidad_negativas, "\n")
cat(sprintf("Porcentaje positivo: %.2f%%\n", porcentaje_palabras(palabras, positivas)))
cat(sprintf("Porcentaje negativo: %.2f%%\n", porcentaje_palabras(palabras, negativas)))
cat("\nPalabras positivas encontradas:\n")
print(table(palabras[palabras %in% positivas]))
cat("\nPalabras negativas encontradas:\n")
print(table(palabras[palabras %in% negativas]))

Apariciones positivas: 20 
Apariciones negativas: 5 
Porcentaje positivo: 26.32%
Porcentaje negativo: 6.58%

Palabras positivas encontradas:

     aprendi         buen        bueno       buenos     completo    excelente 
           2            3            1            1            1            7 
      genial recomendable   recomiendo       supero         util 
           1            1            1            1            1 

Palabras negativas encontradas:

   basico      malo mejorable   mejorar 
        1         2         1         1 


# 4. Clasificación automática

### 4.1. Prueba de las reglas de clasificación

In [44]:
cat(clasificar_comentario("Excelente contenido y profesor", positivas, negativas), "\n")
cat(clasificar_comentario("No lo recomiendo", positivas, negativas), "\n")
cat(clasificar_comentario("Curso bueno, aunque mejorable", positivas, negativas), "\n")

Positivo 
Negativo 
Neutro 


### 4.2. Agrupación en positivos, neutros y negativos

In [45]:
etiquetas <- vapply(comentarios, clasificar_comentario, character(1),
                    positivas = positivas, negativas = negativas,
                    USE.NAMES = FALSE)
etiquetas <- factor(etiquetas, levels = c("Positivo", "Neutro", "Negativo"))

grupos <- split(comentarios, etiquetas)

for (categoria in names(grupos)) {
  lista_comentarios <- grupos[[categoria]]
  porcentaje <- length(lista_comentarios) / length(comentarios) * 100
  cat(sprintf("\n%s: %d (%.2f%%)\n", categoria, length(lista_comentarios), porcentaje))
  for (comentario in lista_comentarios) {
    cat("-", comentario, "\n")
  }
}


Positivo: 11 (61.11%)
- Muy buen curso, volvería a tomarlo 
- Excelente contenido y profesor 
- Muy completo, realmente excelente 
- Aprendí bastante, buen curso 
- Excelente, aprendí cosas nuevas 
- Muy útil para mi trabajo 
- Genial enfoque, excelente! 
- Muy buen curso, buenos ejemplos 
- Excelente de verdad, recomendable 
- Excelente, superó mis expectativas 
- Excelente, volvería a tomarlo 

Neutro: 3 (16.67%)
- Regular, esperaba más ejercicios 
- Regular, temas poco profundizados 
- Curso bueno, aunque mejorable 

Negativo: 4 (22.22%)
- Demasiado básico para mi nivel 
- Muy malo, no lo recomiendo 
- Malo, esperaba otra cosa 
- Podría mejorar bastante 


### 4.3. Tabla cruzada: calificación vs. clasificación del comentario

In [46]:
tabla_cruzada <- table(calificacion = calificaciones, clasificacion = etiquetas)
print(addmargins(tabla_cruzada))

            clasificacion
calificacion Positivo Neutro Negativo Sum
         1          0      0        2   2
         2          0      0        2   2
         3          0      2        0   2
         4          4      1        0   5
         5          7      0        0   7
         Sum       11      3        4  18


### 4.4. Preparación de registros para análisis de sentimiento y modelado posterior

In [47]:
registros <- data.frame(
  id = seq_along(comentarios),
  calificacion = calificaciones,
  comentario = comentarios,
  texto_limpio = comentarios_limpios,
  num_tokens = lengths(tokens_por_comentario),
  positivas = vapply(tokens_por_comentario, contar_palabras, numeric(1), vocabulario = positivas),
  negativas = vapply(tokens_por_comentario, contar_palabras, numeric(1), vocabulario = negativas),
  clasificacion = etiquetas
)
registros$tokens <- tokens_por_comentario   # columna lista con los tokens

cat("Estructura del data.frame (sin la columna lista tokens):\n")
str(registros[, names(registros) != "tokens"])
cat("+ columna tokens: lista de", length(registros$tokens), "vectores de palabras\n")

cat("\nPrimer registro:\n")
for (campo in names(registros)) {
  valor <- registros[[campo]][[1]]
  cat(campo, ":", as.character(valor), "\n")
}

Estructura del data.frame (sin la columna lista tokens):
'data.frame':	18 obs. of  8 variables:
 $ id           : int  1 2 3 4 5 6 7 8 9 10 ...
 $ calificacion : num  4 5 3 5 4 2 1 5 4 5 ...
 $ comentario   : chr  "Muy buen curso, volvería a tomarlo" "Excelente contenido y profesor" "Regular, esperaba más ejercicios" "Muy completo, realmente excelente" ...
 $ texto_limpio : chr  "muy buen curso volveria a tomarlo" "excelente contenido y profesor" "regular esperaba mas ejercicios" "muy completo realmente excelente" ...
 $ num_tokens   : int  6 4 4 4 4 5 5 4 5 3 ...
 $ positivas    : num  1 1 0 2 2 0 1 2 1 2 ...
 $ negativas    : num  0 0 0 0 0 1 1 0 0 0 ...
 $ clasificacion: Factor w/ 3 levels "Positivo","Neutro",..: 1 1 2 1 1 3 3 1 1 1 ...
+ columna tokens: lista de 18 vectores de palabras

Primer registro:
id : 1 
calificacion : 4 
comentario : Muy buen curso, volvería a tomarlo 
texto_limpio : muy buen curso volveria a tomarlo 
num_tokens : 6 
positivas : 1 
negativas : 0 
clasificac

# 5. Recomendación ejecutiva
Con base en el análisis, la percepción de los cursos es mayoritariamente favorable: la calificación media es 3.72, la mediana 4 y la moda 5; el 66.67% de los estudiantes otorgó 4 o 5 y el 61.11% de los comentarios se clasificó como positivo. 

Sin embargo, la varianza poblacional de 1.87 y un 22.22% de calificaciones bajas (1 y 2) revelan un segmento insatisfecho relevante. 

Los comentarios negativos y neutros coinciden en tres causas: contenido demasiado básico, falta de ejercicios prácticos y temas poco profundizados. Se recomienda a la UCC ofrecer rutas diferenciadas por nivel, ampliar los ejercicios aplicados y profundizar los temas clave. Además, conviene ampliar la muestra e integrar este pipeline como monitoreo periódico del impacto de las mejoras.